### Environment Setup

In [ ]:
%pip install datasets tqdm transformers sentence-transformers nltk

### Imports & Model Configuration

In [ ]:
import re
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
from sentence_transformers import SentenceTransformer, util
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from datasets import load_dataset
from tqdm.notebook import tqdm  # Uses the Jupyter-friendly progress bar

# 1. Local HF model config (LLM judge)
MODEL_ID = "unsloth/Meta-Llama-3.1-8B-Instruct-GGUF"
GGUF_FILE = "Meta-Llama-3.1-8B-Instruct-Q4_K_M.gguf"

print("Loading Meta-Llama-3.1-8B-Instruct (GGUF) from Hugging Face...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, gguf_file=GGUF_FILE)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    gguf_file=GGUF_FILE,
    device_map="auto",
    torch_dtype=torch.float16,
)
eval_pipe = pipeline("text-generation", model=model, tokenizer=tokenizer)
print("Local LLM initialized.")

# 2. Lightweight local memory (no API calls)
class LocalMemory:
    def __init__(self, embedding_model="sentence-transformers/all-MiniLM-L6-v2"):
        self.embedder = SentenceTransformer(embedding_model)
        self._store = {}

    def add(self, text, user_id):
        chunks = [line.strip() for line in text.split("\n") if line.strip()]
        if not chunks:
            chunks = [text.strip()] if text.strip() else []
        if not chunks:
            self._store[user_id] = {"chunks": [], "embeddings": None}
            return
        embeddings = self.embedder.encode(chunks, convert_to_tensor=True, normalize_embeddings=True)
        self._store[user_id] = {"chunks": chunks, "embeddings": embeddings}

    def search(self, query, user_id, top_k=5):
        data = self._store.get(user_id)
        if not data or not data["chunks"]:
            return []
        query_emb = self.embedder.encode(query, convert_to_tensor=True, normalize_embeddings=True)
        scores = util.cos_sim(query_emb, data["embeddings"])[0]
        k = min(top_k, len(data["chunks"]))
        top_indices = torch.topk(scores, k=k).indices.tolist()
        return [{"memory": data["chunks"][idx], "score": float(scores[idx])} for idx in top_indices]

    def delete_all(self, user_id):
        self._store.pop(user_id, None)


def normalize_text(text):
    text = text.lower().strip()
    text = re.sub(r"[^a-z0-9\s]", "", text)
    text = re.sub(r"\s+", " ", text)
    return text


def token_f1(prediction, reference):
    pred_tokens = normalize_text(prediction).split()
    ref_tokens = normalize_text(reference).split()
    if not pred_tokens and not ref_tokens:
        return 1.0
    if not pred_tokens or not ref_tokens:
        return 0.0

    pred_counts = {}
    for token in pred_tokens:
        pred_counts[token] = pred_counts.get(token, 0) + 1

    ref_counts = {}
    for token in ref_tokens:
        ref_counts[token] = ref_counts.get(token, 0) + 1

    overlap = 0
    for token, count in pred_counts.items():
        overlap += min(count, ref_counts.get(token, 0))

    if overlap == 0:
        return 0.0

    precision = overlap / len(pred_tokens)
    recall = overlap / len(ref_tokens)
    return 2 * precision * recall / (precision + recall)


def bleu1_score(prediction, reference):
    pred_tokens = normalize_text(prediction).split()
    ref_tokens = normalize_text(reference).split()
    if not pred_tokens or not ref_tokens:
        return 0.0
    return sentence_bleu(
        [ref_tokens],
        pred_tokens,
        weights=(1.0, 0.0, 0.0, 0.0),
        smoothing_function=SmoothingFunction().method1,
    )


def llm_as_judge(question, reference, prediction):
    judge_prompt = f"""
You are a strict evaluator for multiple-choice QA.
Given a question, the ground-truth answer, and a model answer, output ONLY YES or NO.
YES if the model answer matches the same choice/meaning as ground truth.
NO otherwise.

Question: {question}
Ground Truth: {reference}
Model Answer: {prediction}
""".strip()

    outputs = eval_pipe(
        judge_prompt,
        max_new_tokens=3,
        do_sample=False,
        temperature=0.0,
        pad_token_id=tokenizer.eos_token_id,
    )
    generated = outputs[0]["generated_text"][len(judge_prompt):].strip().upper()
    if generated.startswith("YES"):
        return 1
    if generated.startswith("NO"):
        return 0
    return 0


memory = LocalMemory()
print("Local memory initialized (strictly local mode).")

### Data Loading

In [ ]:
print("Loading LoCoMo dataset...")
dataset = load_dataset("Percena/locomo-mc10", split="train")

# Lightweight local test profile for 8GB VRAM.
TEST_SIZE = 3  # Increase gradually to 10/20 once stable.
test_subset = dataset.select(range(TEST_SIZE))

print(f"Loaded {len(test_subset)} examples for lightweight testing.")
print("\nSample Question:", test_subset[0]['question'])

### Evaluation Loop

In [ ]:
correct_predictions = 0
total_questions = len(test_subset)
results_log = []

bleu1_scores = []
f1_scores = []
llm_judge_scores = []

for i, row in enumerate(tqdm(test_subset, desc="Running LoCoMo Baseline")):
    session_id = f"locomo_eval_session_{i}"

    dialogue = row['context']
    question = row['question']
    choices = row['choices']
    ground_truth = row['answer']

    try:
        # PHASE A: Ingestion
        memory.add(dialogue, user_id=session_id)

        # PHASE B: Retrieval
        retrieved_data = memory.search(question, user_id=session_id, top_k=5)
        context_string = "\n".join([item['memory'] for item in retrieved_data])

        # PHASE C: Local LLM Answer Generation
        eval_prompt = f"""
Based ONLY on the following retrieved context, answer the multiple-choice question.

Context:
{context_string}

Question: {question}
Choices: {choices}

Return ONLY the exact text of the correct choice. Do not include introductory text.
""".strip()

        outputs = eval_pipe(
            eval_prompt,
            max_new_tokens=48,
            do_sample=False,
            temperature=0.0,
            pad_token_id=tokenizer.eos_token_id,
        )
        generated_text = outputs[0]["generated_text"]
        model_answer = generated_text[len(eval_prompt):].strip().split("\n")[0]

        # PHASE D: Metrics
        is_correct = (model_answer.lower() == ground_truth.lower())
        if is_correct:
            correct_predictions += 1

        cur_bleu1 = bleu1_score(model_answer, ground_truth)
        cur_f1 = token_f1(model_answer, ground_truth)
        cur_llm_judge = llm_as_judge(question, ground_truth, model_answer)

        bleu1_scores.append(cur_bleu1)
        f1_scores.append(cur_f1)
        llm_judge_scores.append(cur_llm_judge)

        results_log.append({
            "session_id": session_id,
            "question": question,
            "retrieved_context": context_string,
            "model_answer": model_answer,
            "ground_truth": ground_truth,
            "is_correct": is_correct,
            "bleu1": cur_bleu1,
            "f1": cur_f1,
            "llm_judge": cur_llm_judge,
        })

        # Clean up memory
        memory.delete_all(user_id=session_id)

    except Exception as e:
        print(f"\nError processing session {session_id}: {e}")

exact_match_accuracy = (correct_predictions / total_questions) * 100 if total_questions else 0.0
avg_bleu1 = (sum(bleu1_scores) / len(bleu1_scores)) if bleu1_scores else 0.0
avg_f1 = (sum(f1_scores) / len(f1_scores)) if f1_scores else 0.0
llm_judge_rate = (sum(llm_judge_scores) / len(llm_judge_scores)) * 100 if llm_judge_scores else 0.0

print(f"\nExact Match Accuracy: {exact_match_accuracy:.2f}% ({correct_predictions}/{total_questions})")
print(f"Average BLEU-1: {avg_bleu1:.4f}")
print(f"Average F1 Score: {avg_f1:.4f}")
print(f"LLM-as-a-Judge (YES rate): {llm_judge_rate:.2f}%")

### Result Reviews

In [ ]:
# Print out the questions the baseline got wrong
print("--- ERROR ANALYSIS ---")
for log in results_log:
    if not log['is_correct']:
        print(f"\nQuestion: {log['question']}")
        print(f"Expected: {log['ground_truth']}")
        print(f"Model Guessed: {log['model_answer']}")
        print(f"Retrieved Context: {log['retrieved_context'][:200]}...") # Truncated for readability